# El agente de voz por el método sándwich

**Lección 3 · Clase 5.3** — juntamos las dos lecciones anteriores alrededor de un agente y obtenemos algo con lo que se puede *conversar*.

La arquitectura se llama **sándwich** (o cascada) porque el agente de texto va en el medio y las capas de voz son el pan:

```
🎙️  audio  ──▶  STT  ──▶  texto  ──▶  AGENTE  ──▶  texto  ──▶  TTS  ──▶  audio  🔊
              lección 1                (+ herramientas)         lección 2
```

Lo elegante del patrón es que **ninguno de los tres modelos sabe nada de voz-a-voz**. Cada uno hace una cosa y el pegamento es tuyo. Eso trae ventajas reales: puedes elegir el mejor proveedor de cada capa, cambiar uno sin tocar los otros, y —sobre todo— tu agente sigue siendo un agente de **texto**, con todo lo que eso implica: es fácil de testear, de loguear, de evaluar y de razonar sobre él. Todo lo que ya sabes de agentes se aplica sin cambios.

Y trae un problema, que es el tema real de esta lección: **las latencias se suman**. Vamos a medirlas una por una y a ver qué queda.

| | |
|---|---|
| **Capa 2** | El agente de texto con `create_agent` y búsqueda web. Y por qué el modelo se elige por **latencia**. |
| **El sándwich** | Las tres capas encadenadas en una función, instrumentada. |
| **El presupuesto** | Cuánto aporta cada capa al tiempo que el usuario espera. |
| **La interfaz** | Gradio con micrófono: hablarle de verdad. |
| **El techo** | Qué no puede hacer esta arquitectura, por diseño. |

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q langchain==1.3.14 langchain-core==1.5.3 langchain-openai==1.4.1 \
#   langchain-tavily==0.2.18 openai==2.52.0 gradio==6.22.0 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_TAVILY = bool(os.environ.get("TAVILY_API_KEY"))
print("OPENAI_API_KEY presente:", HAY_OPENAI)
print("TAVILY_API_KEY presente:", HAY_TAVILY)
if not HAY_TAVILY:
    print("⚠️ Sin Tavily el agente funciona igual, pero sin buscar en internet")
    print("   (llave gratis en tavily.com: 1.000 búsquedas/mes).")
if not HAY_OPENAI:
    print("⛔ Sin OPENAI_API_KEY esta lección no puede correr: las tres capas son de OpenAI.")

from pathlib import Path

SALIDAS = Path("outputs")
SALIDAS.mkdir(exist_ok=True)

## La capa del medio: el agente

Acá no hay nada nuevo respecto a lo que ya vimos del curso — y ese es justamente el punto del sándwich. Un agente de texto con una herramienta de búsqueda, armado con `create_agent` de LangChain 1.x.

Dos notas sobre el código, porque el material que circula por internet está desactualizado:

- `create_agent` reemplazó al par `create_tool_calling_agent` + `AgentExecutor`. Devuelve un grafo compilado que se invoca con `{"messages": [...]}`.
- La herramienta de Tavily vive ahora en el paquete **`langchain-tavily`** (`TavilySearch`). La vieja `TavilySearchResults` de `langchain_community` está deprecada.

El personaje lo heredamos del material original de la clase: Luis, que habla en chileno. Y el prompt tiene un detalle que importa más de lo que parece: **le pedimos explícitamente que responda como en una conversación hablada**. Un agente de texto por defecto responde con listas, títulos y links — todo eso, leído en voz alta, es insoportable. El prompt de un agente de voz no es el mismo que el de un chat.

In [ ]:
from datetime import date

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

INSTRUCCIONES = f"""
Te llamas Luis y eres un asistente chileno que conversa por voz.

Reglas de la conversación:
- Responde SIEMPRE como si estuvieras hablando, no escribiendo: frases cortas, tono informal
  y cercano, español de Chile.
- Máximo 2 o 3 frases. Si la respuesta es larga, da lo esencial y ofrece profundizar.
- NUNCA uses listas, numeraciones, títulos, markdown, URLs ni emojis: todo esto se va a leer
  en voz alta y esos elementos no se pueden pronunciar.
- Escribe las cifras como se dicen ("novecientos treinta pesos", no "$930").
- Si no sabes algo y tienes búsqueda disponible, búscalo antes de responder.

La fecha de hoy es {date.today().strftime('%d/%m/%Y')}.
"""

herramientas = []
if HAY_TAVILY:
    from langchain_tavily import TavilySearch

    herramientas.append(TavilySearch(max_results=3, country="chile"))

print("Herramientas del agente:", [h.name for h in herramientas] or "ninguna")

### El modelo se elige por latencia, no por inteligencia

Acá está la decisión de diseño más importante de la lección, y es contraintuitiva.

En un chat escrito, que el modelo se demore diez segundos pensando es aceptable: el usuario ve un indicador y espera. En una conversación hablada, **diez segundos de silencio es una llamada caída**. La referencia dura viene de la lingüística: en una conversación humana el turno cambia en unos **200 ms**, y sobre ~1 segundo de silencio la gente empieza a asumir que algo se cortó y a hablar encima.

Eso descalifica de entrada a los modelos con razonamiento extendido, por buenos que sean. Midámoslo en vez de suponerlo: la misma pregunta, que obliga a buscar en internet, contra cuatro configuraciones.

In [ ]:
import time

PREGUNTA_BENCH = "busca en internet: cuánto está el dólar hoy en Chile?"

CANDIDATOS = {
    "gpt-5-mini (razonamiento por defecto)": ChatOpenAI(model="gpt-5-mini"),
    "gpt-5-mini (reasoning_effort=minimal)": ChatOpenAI(model="gpt-5-mini", reasoning_effort="minimal"),
    "gpt-5-nano (reasoning_effort=minimal)": ChatOpenAI(model="gpt-5-nano", reasoning_effort="minimal"),
    "gpt-4.1-mini (sin razonamiento)": ChatOpenAI(model="gpt-4.1-mini"),
}


def medir_agente(modelo) -> tuple[float, int, str]:
    """Devuelve (segundos, nº de llamadas a herramienta, respuesta)."""
    agente = create_agent(modelo, tools=herramientas, system_prompt=INSTRUCCIONES)
    inicio = time.perf_counter()
    resultado = agente.invoke({"messages": [{"role": "user", "content": PREGUNTA_BENCH}]})
    transcurrido = time.perf_counter() - inicio
    usos = sum(1 for m in resultado["messages"] if type(m).__name__ == "ToolMessage")
    return transcurrido, usos, resultado["messages"][-1].content


if HAY_OPENAI:
    print(f"Pregunta: {PREGUNTA_BENCH}\n")
    print(f"{'modelo':<40} {'tiempo':>8} {'tools':>6}")
    print("-" * 58)
    for nombre, modelo in CANDIDATOS.items():
        try:
            segundos, usos, respuesta = medir_agente(modelo)
            print(f"{nombre:<40} {segundos:>7.1f}s {usos:>6}")
            print(f"    → {respuesta[:150]}")
        except Exception as error:
            print(f"{nombre:<40} error: {type(error).__name__}")
else:
    print("⛔ Falta OPENAI_API_KEY.")

El ordenamiento se repite cada vez que corremos esto, y es de **un orden de magnitud**:

- **`gpt-5-mini` con razonamiento por defecto es inviable por teléfono.** Se toma decenas de segundos y encadena muchas búsquedas. En un chat escrito sería una respuesta cuidadosa; en una llamada, el cliente ya colgó.
- **Bajar `reasoning_effort` ayuda pero no alcanza.** Sigue muy lejos del presupuesto.
- **Los modelos sin razonamiento extendido son los que caben.** `gpt-4.1-mini` responde en pocos segundos.

Y ahora la parte incómoda, que es la que hace interesante la decisión: **el modelo lento suele dar la mejor respuesta**. Preparando esta lección, en varias corridas `gpt-5-mini` fue el único que consiguió el valor exacto del dólar — insistió con nueve búsquedas hasta encontrarlo — mientras `gpt-4.1-mini` se dio por vencido después de una y ofreció ayudar de otra forma. Esa tenacidad es exactamente lo que el razonamiento compra, y es exactamente lo que no cabe en un presupuesto de voz.

Entonces esto **no es un almuerzo gratis**: estás cambiando calidad de respuesta por tiempo de respuesta, y en voz el tiempo gana casi siempre porque una respuesta perfecta que llega en un minuto ya no es una respuesta. La forma adulta de resolverlo en producción es no elegir uno: contestar rápido con el modelo chico ("déjame revisar eso") y escalar al grande en segundo plano cuando la pregunta lo amerite.

La moraleja general: **el mejor modelo depende de la restricción, no del ranking**. La misma familia `gpt-5` que elegiríamos sin dudar para el OCR semántico de la clase 5.2 —donde nadie está esperando— es la peor opción acá.

> Cuidado con lo que *no* dice esta medición: es una sola pregunta, una corrida por modelo, y las latencias de API varían con la hora y la carga. Lo robusto es el orden de magnitud y el ordenamiento; los segundos exactos y quién acierta el dato van a cambiar en tu corrida.

Nos quedamos con `gpt-4.1-mini` para el resto de la lección.

In [ ]:
MODELO_AGENTE = "gpt-4.1-mini"
MODELO_STT = "gpt-transcribe"
MODELO_TTS = "gpt-4o-mini-tts"
VOZ = "marin"

agente = create_agent(
    ChatOpenAI(model=MODELO_AGENTE),
    tools=herramientas,
    system_prompt=INSTRUCCIONES,
) if HAY_OPENAI else None

print(f"Sándwich armado:  {MODELO_STT}  →  {MODELO_AGENTE}  →  {MODELO_TTS} (voz {VOZ})")

## Las tres capas, encadenadas

Ahora el pegamento. Tres funciones cortas —una por capa— y una cuarta que las encadena midiendo cada tramo. La instrumentación no es decoración: en un sistema de voz, saber **cuál** capa se está demorando es la diferencia entre optimizar y adivinar.

In [ ]:
from openai import OpenAI

cliente = OpenAI() if HAY_OPENAI else None


def escuchar(ruta_audio: str) -> str:
    """Capa 1 — audio a texto."""
    with open(ruta_audio, "rb") as archivo:
        return cliente.audio.transcriptions.create(model=MODELO_STT, file=archivo).text


def pensar(texto: str) -> str:
    """Capa 2 — el agente de texto responde (y busca si hace falta)."""
    resultado = agente.invoke({"messages": [{"role": "user", "content": texto}]})
    return resultado["messages"][-1].content


def responder(texto: str, nombre_archivo: str) -> Path:
    """Capa 3 — texto a audio."""
    destino = SALIDAS / nombre_archivo
    with cliente.audio.speech.with_streaming_response.create(
        model=MODELO_TTS, voice=VOZ, input=texto, response_format="mp3",
        instructions="Habla como un chileno conversando por teléfono: cercano, natural, ritmo normal.",
    ) as respuesta:
        respuesta.stream_to_file(destino)
    return destino


def conversar(ruta_audio: str, etiqueta: str = "turno") -> dict:
    """El sándwich completo, con el tiempo de cada capa."""
    tiempos = {}

    inicio = time.perf_counter()
    pregunta = escuchar(ruta_audio)
    tiempos["1. escuchar (STT)"] = time.perf_counter() - inicio

    inicio = time.perf_counter()
    respuesta_texto = pensar(pregunta)
    tiempos["2. pensar (agente)"] = time.perf_counter() - inicio

    inicio = time.perf_counter()
    ruta_respuesta = responder(respuesta_texto, f"{etiqueta}_respuesta.mp3")
    tiempos["3. responder (TTS)"] = time.perf_counter() - inicio

    return {
        "pregunta": pregunta,
        "respuesta": respuesta_texto,
        "audio": ruta_respuesta,
        "tiempos": tiempos,
    }


print("Capas definidas: escuchar() → pensar() → responder(), y conversar() las encadena.")

### Un turno completo

Para probarlo sin micrófono necesitamos audio de entrada, así que lo **fabricamos con el TTS de la lección 2**: sintetizamos la pregunta del usuario y se la damos de comer al sándwich. Es un truco útil más allá de la clase — sirve para armar tests automatizados de un agente de voz sin grabar a nadie.

In [ ]:
from IPython.display import Audio, display

PREGUNTA_USUARIO = "Hola Luis, cuéntame qué se puede hacer hoy con inteligencia artificial en una empresa chilena."

turno = None

if not HAY_OPENAI:
    print("⛔ Falta OPENAI_API_KEY.")
else:
    # Fabricamos la "grabación" del usuario con otra voz, para que no sea la misma que responde
    entrada = SALIDAS / "turno1_pregunta.mp3"
    with cliente.audio.speech.with_streaming_response.create(
        model=MODELO_TTS, voice="onyx", input=PREGUNTA_USUARIO, response_format="mp3",
    ) as respuesta:
        respuesta.stream_to_file(entrada)

    print("🎙️  Lo que 'dijo' el usuario:")
    display(Audio(filename=str(entrada)))

    turno = conversar(str(entrada), etiqueta="turno1")

    print(f"\n📝 Transcrito : {turno['pregunta']}")
    print(f"🤖 Luis       : {turno['respuesta']}")
    print("\n🔊 La respuesta hablada:")
    display(Audio(filename=str(turno["audio"])))

## El presupuesto de latencia

Acá se ve el costo de la arquitectura. Cada capa aporta su parte, y el usuario espera la **suma**.

In [ ]:
if turno:
    total = sum(turno["tiempos"].values())

    print(f"{'capa':<24} {'tiempo':>8} {'% del total':>12}")
    print("-" * 48)
    for capa, segundos in turno["tiempos"].items():
        print(f"{capa:<24} {segundos:>7.2f}s {segundos / total * 100:>11.0f}%")
    print("-" * 48)
    print(f"{'TOTAL que espera el usuario':<24} {total:>7.2f}s")

    print()
    if total > 2:
        print(f"⚠️  {total:.1f} segundos de silencio. En una conversación humana el turno")
        print("    cambia en ~0,2 s; sobre ~1 s la gente cree que se cortó la llamada.")
    print("\nY esto es el mejor caso: sin ruido, sin interrupciones, una sola pregunta corta.")
else:
    print("(sin llave no hay medición)")

### Qué se puede hacer con esto, y qué no

Fíjate primero en **dónde** se fue el tiempo, porque suele sorprender: la capa que más pesa no es el agente, es el **TTS**. Es un buen recordatorio de que la intuición ("lo caro es pensar") falla acá, y de que sin instrumentar cada capa habrías optimizado la equivocada.

El presupuesto se puede apretar bastante, y vale saber cómo:

- **Streaming en las tres capas.** Es la optimización grande. No esperes a que termine la transcripción para empezar a pensar, ni a que termine el texto para empezar a sintetizar: se puede ir hablando mientras el agente todavía escribe. Bien hecho, el usuario oye la primera sílaba en cientos de milisegundos en vez de segundos. Es más código, pero es *el* trabajo de un agente de voz en producción.
- **Un TTS `flash`.** Dado que la capa 3 es la que más pesa, es la primera que hay que atacar: en la lección 2 medimos `eleven_flash_v2_5` unas 5 veces más rápido que los modelos expresivos, y cambiar esta capa es una línea de código.
- **Un modelo más chico y un prompt más corto.** Cada token de salida es tiempo. Las respuestas de 2-3 frases que le pedimos a Luis no son solo estilo: son presupuesto.
- **Evitar las herramientas cuando se pueda.** Cada búsqueda es un viaje de ida y vuelta extra dentro de la capa 2.

Pero hay un techo que **no** se puede optimizar, porque no está en la implementación sino en la arquitectura:

- **La prosodia del usuario se pierde en la transcripción.** El STT entrega texto plano: "está todo bien" sale igual dicho con alivio o con ironía furiosa. Toda la información de *cómo* se dijo algo —el enojo, la duda, la urgencia, el sarcasmo— muere en la capa 1 y el agente nunca la ve. Para un canal de atención al cliente, eso es exactamente lo que más importaba.
- **Interrumpir es muy difícil.** El sistema está sintetizando un audio completo; para que el usuario pueda cortarlo hay que detectar que empezó a hablar, abortar la reproducción, descartar el turno y rearmar el contexto. Todo eso es plomería que tú tienes que escribir.
- **Los turnos son rígidos.** Hablas, esperas, escuchas. No hay "mmm", ni asentir mientras el otro habla, ni las superposiciones naturales de una conversación real.

Esos tres límites son la razón de existir de la **lección 4**: modelos que van de audio a audio sin pasar por texto.

## La interfaz: hablarle de verdad

Todo lo anterior fue con audio sintético. Para hablarle con tu propio micrófono, Gradio.

> El material original de la clase usaba `ipywebrtc` para grabar dentro del notebook; hoy no funciona en los entornos actuales de Jupyter/Colab. Gradio resuelve lo mismo y además funciona igual en local y en Colab.

Levantamos el servidor con `prevent_thread_lock=True` para que la celda no quede bloqueada. **Abre la URL que imprime**, graba, y verás el texto y la respuesta hablada.

In [ ]:
import gradio as gr

interfaz = None


def turno_de_voz(ruta_audio):
    """Handler de Gradio: audio del micrófono → (texto, audio de respuesta)."""
    if not ruta_audio:
        return "No se grabó nada.", None

    resultado = conversar(ruta_audio, etiqueta="gradio")
    total = sum(resultado["tiempos"].values())
    detalle = " · ".join(f"{c.split('. ')[1]} {s:.1f}s" for c, s in resultado["tiempos"].items())

    texto = (
        f"Tú dijiste: {resultado['pregunta']}\n\n"
        f"Luis: {resultado['respuesta']}\n\n"
        f"⏱️ {total:.1f}s en total  ({detalle})"
    )
    return texto, str(resultado["audio"])


if HAY_OPENAI:
    interfaz = gr.Interface(
        fn=turno_de_voz,
        inputs=gr.Audio(sources=["microphone"], type="filepath", label="Habla con Luis"),
        outputs=[
            gr.Textbox(label="Transcripción, respuesta y latencias", lines=8),
            gr.Audio(type="filepath", label="Luis responde", autoplay=True),
        ],
        title="Agente de voz — método sándwich",
        description=f"{MODELO_STT} → {MODELO_AGENTE} → {MODELO_TTS}",
        flagging_mode="never",
    )

    interfaz.launch(prevent_thread_lock=True, quiet=True)
    print(f"🌐 Abre: {interfaz.local_url}")
    print("   (cuando termines, corre la celda siguiente para cerrar el servidor)")
else:
    print("⛔ Falta OPENAI_API_KEY.")

In [ ]:
# Cierra el servidor de Gradio y libera el puerto
if interfaz is not None:
    interfaz.close()
    print("Servidor cerrado.")

## Qué nos llevamos

- El **sándwich** (STT → agente → TTS) es la forma más simple y más portable de construir un agente de voz, y su gran virtud es que **el agente sigue siendo de texto**: testeable, logueable, evaluable, y con todos los patrones de agentes que ya conoces.
- El **prompt de un agente de voz es distinto** al de un chat: sin listas, sin markdown, sin URLs, respuestas de dos o tres frases, cifras escritas como se pronuncian.
- El modelo del medio se elige por **latencia**. Medimos que un modelo con razonamiento extendido queda fuera por decenas de segundos, mientras uno sin razonamiento responde en pocos. El mejor modelo depende de la restricción, no del ranking.
- Las latencias **se suman**, y el presupuesto de referencia es brutal: ~200 ms es el turno humano. El streaming en las tres capas es la optimización que de verdad mueve la aguja.
- Hay tres límites que la arquitectura no puede superar: **se pierde la prosodia** del usuario, **interrumpir es plomería difícil**, y **los turnos son rígidos**.

En la **lección 4** atacamos justo eso: modelos *voice-to-voice* que reciben audio y devuelven audio sin pasar por texto — y tres frameworks distintos para construir con ellos.